# Advanced Problems with Solutions: `itertools` Combinatorics

This notebook contains advanced practice problems with complete solutions for Python's combinatorics-focused `itertools` tools:

- `itertools.product`
- `itertools.permutations`
- `itertools.combinations`
- `itertools.combinations_with_replacement`

The exercises are designed around real patterns: grids, parameter searches, exact probabilities, cards, dice, seating arrangements, multiset permutations, and constrained selection.

> Best-practice theme: keep iterators lazy, materialize only small previews, use exact arithmetic when possible, and verify results with assertions.


## Setup

Run this cell first. The helper functions make later examples easier to inspect without accidentally materializing huge iterators.


In [1]:
import itertools as it
import math
from collections import Counter, defaultdict, namedtuple
from fractions import Fraction
from functools import reduce
from operator import mul
from pprint import pprint


def take(n, iterable):
    """Return a list containing the first n items from an iterable."""
    return list(it.islice(iterable, n))


def consume_count(iterable):
    """Count items from any iterable without storing the items."""
    return sum(1 for _ in iterable)


def product_size(*iterables):
    """Return the cartesian-product size when each iterable has a known length."""
    return math.prod(len(x) for x in iterables)


def assert_same_items(actual, expected):
    assert sorted(actual) == sorted(expected), f"actual={actual!r}, expected={expected!r}"

print("Setup complete.")


Setup complete.


## Problem 1 — Cartesian product without nested loops

Write a function `multiplication_records(n)` that lazily yields triples `(i, j, i*j)` for all `1 <= i <= n` and `1 <= j <= n`.

Requirements:

1. Use `itertools.product`.
2. Do not build the full table internally.
3. Preserve row-major order: `(1, 1)`, `(1, 2)`, ..., `(2, 1)`, ...


### Solution 1


In [2]:
def multiplication_records(n):
    if n < 1:
        return iter(())
    return ((i, j, i * j) for i, j in it.product(range(1, n + 1), repeat=2))

preview = take(10, multiplication_records(4))
preview


[(1, 1, 1),
 (1, 2, 2),
 (1, 3, 3),
 (1, 4, 4),
 (2, 1, 2),
 (2, 2, 4),
 (2, 3, 6),
 (2, 4, 8),
 (3, 1, 3),
 (3, 2, 6)]

In [3]:
assert take(5, multiplication_records(3)) == [
    (1, 1, 1),
    (1, 2, 2),
    (1, 3, 3),
    (2, 1, 2),
    (2, 2, 4),
]
assert consume_count(multiplication_records(10)) == 100
print("Problem 1 tests passed.")


Problem 1 tests passed.


### Extra example: formatting a lazy multiplication table


In [4]:
def formatted_multiplication_table(n):
    for i, j, value in multiplication_records(n):
        yield f"{i:>2} x {j:<2} = {value:>3}"

print("\n".join(take(12, formatted_multiplication_table(5))))


 1 x 1  =   1
 1 x 2  =   2
 1 x 3  =   3
 1 x 4  =   4
 1 x 5  =   5
 2 x 1  =   2
 2 x 2  =   4
 2 x 3  =   6
 2 x 4  =   8
 2 x 5  =  10
 3 x 1  =   3
 3 x 2  =   6


## Problem 2 — Lazy coordinate grids with exact decimal steps

The original lesson used `count`, `takewhile`, `tee`, and `product` to generate coordinate grids. Floating-point steps such as `0.1` can introduce rounding artifacts.

Write `fraction_grid(min_val, max_val, step, dimensions)` that yields points as tuples of `Fraction` values.

Example:

```python
list(fraction_grid(-1, 1, Fraction(1, 2), 2))
```

should include points from `(-1, -1)` through `(1, 1)` in row-major Cartesian-product order.


### Solution 2


In [5]:
def fraction_axis(min_val, max_val, step):
    """Yield Fraction values from min_val to max_val inclusive."""
    min_val = Fraction(min_val)
    max_val = Fraction(max_val)
    step = Fraction(step)
    if step <= 0:
        raise ValueError("step must be positive")

    current = min_val
    while current <= max_val:
        yield current
        current += step


def fraction_grid(min_val, max_val, step, dimensions=2):
    """Yield an exact n-dimensional grid using Fraction coordinates."""
    if dimensions < 1:
        raise ValueError("dimensions must be at least 1")
    axis = tuple(fraction_axis(min_val, max_val, step))
    return it.product(axis, repeat=dimensions)

points_2d = list(fraction_grid(-1, 1, Fraction(1, 2), dimensions=2))
points_2d[:8], points_2d[-3:]


([(Fraction(-1, 1), Fraction(-1, 1)),
  (Fraction(-1, 1), Fraction(-1, 2)),
  (Fraction(-1, 1), Fraction(0, 1)),
  (Fraction(-1, 1), Fraction(1, 2)),
  (Fraction(-1, 1), Fraction(1, 1)),
  (Fraction(-1, 2), Fraction(-1, 1)),
  (Fraction(-1, 2), Fraction(-1, 2)),
  (Fraction(-1, 2), Fraction(0, 1))],
 [(Fraction(1, 1), Fraction(0, 1)),
  (Fraction(1, 1), Fraction(1, 2)),
  (Fraction(1, 1), Fraction(1, 1))])

In [6]:
assert len(points_2d) == 25
assert points_2d[0] == (Fraction(-1, 1), Fraction(-1, 1))
assert points_2d[-1] == (Fraction(1, 1), Fraction(1, 1))
assert len(list(fraction_grid(0, 1, Fraction(1, 4), dimensions=3))) == 5 ** 3
print("Problem 2 tests passed.")


Problem 2 tests passed.


### Extra example: convert exact grid points to floats only at the boundary


In [7]:
exact_points = take(6, fraction_grid(0, 1, Fraction(1, 3), dimensions=2))
float_points = [tuple(float(coord) for coord in point) for point in exact_points]

print("Exact:")
pprint(exact_points)
print("\nFloat view:")
pprint(float_points)


Exact:
[(Fraction(0, 1), Fraction(0, 1)),
 (Fraction(0, 1), Fraction(1, 3)),
 (Fraction(0, 1), Fraction(2, 3)),
 (Fraction(0, 1), Fraction(1, 1)),
 (Fraction(1, 3), Fraction(0, 1)),
 (Fraction(1, 3), Fraction(1, 3))]

Float view:
[(0.0, 0.0),
 (0.0, 0.3333333333333333),
 (0.0, 0.6666666666666666),
 (0.0, 1.0),
 (0.3333333333333333, 0.0),
 (0.3333333333333333, 0.3333333333333333)]


## Problem 3 — Hyperparameter search grid with pruning

You are given a dictionary of options for a machine-learning experiment. Generate configurations lazily using `itertools.product`.

Then apply these pruning rules:

1. `penalty='l1'` is valid only with `solver='liblinear'`.
2. `learning_rate` matters only when `model='sgd'`; for other models, keep only `learning_rate=None`.
3. `max_depth` matters only when `model='tree'`; for other models, keep only `max_depth=None`.

Return dictionaries, not tuples.


### Solution 3


In [8]:
param_grid = {
    "model": ["logreg", "sgd", "tree"],
    "solver": ["lbfgs", "liblinear"],
    "penalty": ["l1", "l2"],
    "learning_rate": [None, 0.001, 0.01],
    "max_depth": [None, 3, 5],
}


def raw_configs(grid):
    keys = tuple(grid)
    for values in it.product(*(grid[key] for key in keys)):
        yield dict(zip(keys, values))


def is_valid_config(cfg):
    if cfg["penalty"] == "l1" and cfg["solver"] != "liblinear":
        return False
    if cfg["model"] != "sgd" and cfg["learning_rate"] is not None:
        return False
    if cfg["model"] != "tree" and cfg["max_depth"] is not None:
        return False
    return True


def pruned_configs(grid):
    return filter(is_valid_config, raw_configs(grid))

preview = take(8, pruned_configs(param_grid))
pprint(preview)


[{'learning_rate': None,
  'max_depth': None,
  'model': 'logreg',
  'penalty': 'l2',
  'solver': 'lbfgs'},
 {'learning_rate': None,
  'max_depth': None,
  'model': 'logreg',
  'penalty': 'l1',
  'solver': 'liblinear'},
 {'learning_rate': None,
  'max_depth': None,
  'model': 'logreg',
  'penalty': 'l2',
  'solver': 'liblinear'},
 {'learning_rate': None,
  'max_depth': None,
  'model': 'sgd',
  'penalty': 'l2',
  'solver': 'lbfgs'},
 {'learning_rate': 0.001,
  'max_depth': None,
  'model': 'sgd',
  'penalty': 'l2',
  'solver': 'lbfgs'},
 {'learning_rate': 0.01,
  'max_depth': None,
  'model': 'sgd',
  'penalty': 'l2',
  'solver': 'lbfgs'},
 {'learning_rate': None,
  'max_depth': None,
  'model': 'sgd',
  'penalty': 'l1',
  'solver': 'liblinear'},
 {'learning_rate': 0.001,
  'max_depth': None,
  'model': 'sgd',
  'penalty': 'l1',
  'solver': 'liblinear'}]


In [9]:
raw_count = product_size(*param_grid.values())
valid = list(pruned_configs(param_grid))

assert raw_count == 3 * 2 * 2 * 3 * 3
assert len(valid) == 21
assert all(is_valid_config(cfg) for cfg in valid)
assert all(isinstance(cfg, dict) for cfg in valid)

print(f"Raw configurations:   {raw_count}")
print(f"Pruned configurations:{len(valid)}")
print("Problem 3 tests passed.")


Raw configurations:   108
Pruned configurations:21
Problem 3 tests passed.


### Extra example: group valid configurations by model


In [10]:
by_model = defaultdict(list)
for cfg in pruned_configs(param_grid):
    by_model[cfg["model"]].append(cfg)

for model, configs in sorted(by_model.items()):
    print(f"{model:>6}: {len(configs)} configurations")


logreg: 3 configurations
   sgd: 9 configurations
  tree: 9 configurations


## Problem 4 — Dice distribution with exact probabilities

Use `itertools.product` to compute the distribution of sums when rolling `dice` fair dice, each with faces `1..sides`.

Write `dice_sum_distribution(dice, sides)` returning a dictionary:

```python
{
    sum_value: Fraction(number_of_outcomes, total_outcomes),
    ...
}
```

The result must use exact `Fraction` values.


### Solution 4


In [11]:
def dice_sum_distribution(dice, sides=6):
    if dice < 1:
        raise ValueError("dice must be at least 1")
    if sides < 2:
        raise ValueError("sides must be at least 2")

    outcomes = it.product(range(1, sides + 1), repeat=dice)
    counts = Counter(map(sum, outcomes))
    total = sides ** dice
    return {total_sum: Fraction(count, total) for total_sum, count in sorted(counts.items())}

three_dice = dice_sum_distribution(3, 6)
three_dice


{3: Fraction(1, 216),
 4: Fraction(1, 72),
 5: Fraction(1, 36),
 6: Fraction(5, 108),
 7: Fraction(5, 72),
 8: Fraction(7, 72),
 9: Fraction(25, 216),
 10: Fraction(1, 8),
 11: Fraction(1, 8),
 12: Fraction(25, 216),
 13: Fraction(7, 72),
 14: Fraction(5, 72),
 15: Fraction(5, 108),
 16: Fraction(1, 36),
 17: Fraction(1, 72),
 18: Fraction(1, 216)}

In [12]:
assert three_dice[3] == Fraction(1, 216)
assert three_dice[10] == Fraction(27, 216)
assert three_dice[18] == Fraction(1, 216)
assert sum(three_dice.values(), Fraction(0, 1)) == Fraction(1, 1)
print("Problem 4 tests passed.")


Problem 4 tests passed.


### Extra example: most likely sums


In [13]:
def most_likely_sums(dice, sides=6, n=5):
    distribution = dice_sum_distribution(dice, sides)
    return sorted(distribution.items(), key=lambda item: (-item[1], item[0]))[:n]

for total_sum, probability in most_likely_sums(4, 6, n=7):
    print(f"sum={total_sum:>2}: probability={probability} ≈ {float(probability):.4f}")


sum=14: probability=73/648 ≈ 0.1127
sum=13: probability=35/324 ≈ 0.1080
sum=15: probability=35/324 ≈ 0.1080
sum=12: probability=125/1296 ≈ 0.0965
sum=16: probability=125/1296 ≈ 0.0965
sum=11: probability=13/162 ≈ 0.0802
sum=17: probability=13/162 ≈ 0.0802


## Problem 5 — Conditional dice probability

For three six-sided dice, compute the exact probability that:

1. the sum is at least 12, and
2. at least two dice show the same face.

Use `itertools.product`, not a formula.


### Solution 5


In [14]:
def has_repeated_face(outcome):
    return len(set(outcome)) < len(outcome)


def conditional_dice_probability():
    sample_space = it.product(range(1, 7), repeat=3)
    total = 0
    acceptable = 0

    for outcome in sample_space:
        total += 1
        if sum(outcome) >= 12 and has_repeated_face(outcome):
            acceptable += 1

    return Fraction(acceptable, total)

prob = conditional_dice_probability()
prob, float(prob)


(Fraction(13, 72), 0.18055555555555555)

In [15]:
assert prob == Fraction(13, 72)
print(f"Probability = {prob} ≈ {float(prob):.4f}")
print("Problem 5 tests passed.")


Probability = 13/72 ≈ 0.1806
Problem 5 tests passed.


### Extra example: list the first acceptable outcomes


In [16]:
acceptable_outcomes = (
    outcome
    for outcome in it.product(range(1, 7), repeat=3)
    if sum(outcome) >= 12 and has_repeated_face(outcome)
)

take(15, acceptable_outcomes)


[(1, 6, 6),
 (2, 5, 5),
 (2, 6, 6),
 (3, 3, 6),
 (3, 5, 5),
 (3, 6, 3),
 (3, 6, 6),
 (4, 4, 4),
 (4, 4, 5),
 (4, 4, 6),
 (4, 5, 4),
 (4, 5, 5),
 (4, 6, 4),
 (4, 6, 6),
 (5, 2, 5)]

## Problem 6 — Permutations: seating with forbidden adjacency

Six people are attending dinner:

```python
people = ('Ada', 'Ben', 'Cal', 'Dee', 'Eli', 'Fay')
```

Count the number of linear seating arrangements where `Ada` and `Ben` are **not adjacent**.

Then write a generator that yields the valid arrangements lazily.


### Solution 6


In [17]:
people = ("Ada", "Ben", "Cal", "Dee", "Eli", "Fay")


def are_adjacent(arrangement, a, b):
    positions = {person: idx for idx, person in enumerate(arrangement)}
    return abs(positions[a] - positions[b]) == 1


def valid_seatings(people, forbidden_pair=("Ada", "Ben")):
    a, b = forbidden_pair
    return (
        arrangement
        for arrangement in it.permutations(people)
        if not are_adjacent(arrangement, a, b)
    )

valid_count = consume_count(valid_seatings(people))
valid_count


480

In [18]:
# Formula check:
# Total arrangements: 6!
# Adjacent Ada-Ben block: 2 internal orders * 5! block arrangements
expected = math.factorial(6) - 2 * math.factorial(5)
assert valid_count == expected == 480
print("Problem 6 tests passed.")


Problem 6 tests passed.


### Extra example: preview valid seatings


In [19]:
take(5, valid_seatings(people))


[('Ada', 'Cal', 'Ben', 'Dee', 'Eli', 'Fay'),
 ('Ada', 'Cal', 'Ben', 'Dee', 'Fay', 'Eli'),
 ('Ada', 'Cal', 'Ben', 'Eli', 'Dee', 'Fay'),
 ('Ada', 'Cal', 'Ben', 'Eli', 'Fay', 'Dee'),
 ('Ada', 'Cal', 'Ben', 'Fay', 'Dee', 'Eli')]

## Problem 7 — Circular permutations by canonical rotation

In a circular table arrangement, rotations are considered the same. For example:

```python
('Ada', 'Ben', 'Cal')
('Ben', 'Cal', 'Ada')
('Cal', 'Ada', 'Ben')
```

represent the same circular order.

Write `circular_permutations(items)` that fixes the first item and permutes the rest.

Then count circular arrangements of six people where `Ada` and `Ben` are not adjacent around the circle.


### Solution 7


In [20]:
def circular_permutations(items):
    items = tuple(items)
    if not items:
        return iter(())
    first, rest = items[0], items[1:]
    return ((first,) + tail for tail in it.permutations(rest))


def circular_neighbors(arrangement, person):
    idx = arrangement.index(person)
    return arrangement[idx - 1], arrangement[(idx + 1) % len(arrangement)]


def circular_adjacent(arrangement, a, b):
    return b in circular_neighbors(arrangement, a)

valid_circular = [
    arrangement
    for arrangement in circular_permutations(people)
    if not circular_adjacent(arrangement, "Ada", "Ben")
]

len(valid_circular), valid_circular[:5]


(72,
 [('Ada', 'Cal', 'Ben', 'Dee', 'Eli', 'Fay'),
  ('Ada', 'Cal', 'Ben', 'Dee', 'Fay', 'Eli'),
  ('Ada', 'Cal', 'Ben', 'Eli', 'Dee', 'Fay'),
  ('Ada', 'Cal', 'Ben', 'Eli', 'Fay', 'Dee'),
  ('Ada', 'Cal', 'Ben', 'Fay', 'Dee', 'Eli')])

In [21]:
# Total circular arrangements: (6-1)! = 120
# If Ada and Ben are adjacent in a circle, treat them as a block:
# 2 internal orders * (5-1)! circular arrangements = 48
assert len(valid_circular) == math.factorial(5) - 2 * math.factorial(4) == 72
print("Problem 7 tests passed.")


Problem 7 tests passed.


## Problem 8 — Unique permutations of a multiset

`itertools.permutations('AAB')` returns 6 tuples because elements are treated as unique by position, not by value.

Write two functions:

1. `unique_permutations_small(iterable, r=None)` — simple solution using `dict.fromkeys` over `itertools.permutations`. Good for small inputs.
2. `unique_permutations_counter(iterable)` — efficient full-length unique permutations using `Counter`, without generating duplicate permutations first.


### Solution 8A — Small-input solution


In [22]:
def unique_permutations_small(iterable, r=None):
    return dict.fromkeys(it.permutations(iterable, r)).keys()

list(unique_permutations_small("AAB"))


[('A', 'A', 'B'), ('A', 'B', 'A'), ('B', 'A', 'A')]

### Solution 8B — Counter-based full-length solution


In [23]:
def unique_permutations_counter(iterable):
    counts = Counter(iterable)
    n = sum(counts.values())
    path = []

    def backtrack():
        if len(path) == n:
            yield tuple(path)
            return

        for value in list(counts):
            if counts[value] == 0:
                continue
            counts[value] -= 1
            path.append(value)
            yield from backtrack()
            path.pop()
            counts[value] += 1

    yield from backtrack()

unique_aabc = list(unique_permutations_counter("AABC"))
unique_aabc[:10], len(unique_aabc)


([('A', 'A', 'B', 'C'),
  ('A', 'A', 'C', 'B'),
  ('A', 'B', 'A', 'C'),
  ('A', 'B', 'C', 'A'),
  ('A', 'C', 'A', 'B'),
  ('A', 'C', 'B', 'A'),
  ('B', 'A', 'A', 'C'),
  ('B', 'A', 'C', 'A'),
  ('B', 'C', 'A', 'A'),
  ('C', 'A', 'A', 'B')],
 12)

In [24]:
assert list(unique_permutations_small("AAB")) == [("A", "A", "B"), ("A", "B", "A"), ("B", "A", "A")]
assert len(list(unique_permutations_counter("AABC"))) == math.factorial(4) // math.factorial(2)
assert_same_items(unique_permutations_counter("AAB"), unique_permutations_small("AAB"))
print("Problem 8 tests passed.")


Problem 8 tests passed.


### Extra example: count unique anagrams without generating them


In [25]:
def multiset_permutation_count(iterable):
    counts = Counter(iterable)
    denominator = math.prod(math.factorial(count) for count in counts.values())
    return math.factorial(sum(counts.values())) // denominator

for word in ["LEVEL", "BANANA", "MISSISSIPPI"]:
    print(f"{word:>12}: {multiset_permutation_count(word):>8} unique permutations")


       LEVEL:       30 unique permutations
      BANANA:       60 unique permutations
 MISSISSIPPI:    34650 unique permutations


## Problem 9 — Combinations: constrained team selection

You have candidates with skills. Choose teams of size 4 that satisfy all rules:

1. At least one person knows Python.
2. At least one person knows SQL.
3. At least one person knows ML.
4. The team must include either `Nia` or `Omar`, but not both.

Use `itertools.combinations`.


### Solution 9


In [26]:
candidates = {
    "Nia": {"python", "sql"},
    "Omar": {"ml", "python"},
    "Pia": {"sql", "viz"},
    "Quin": {"ml", "stats"},
    "Rae": {"python", "viz"},
    "Sol": {"sql", "ml"},
    "Tao": {"python", "stats"},
}

required_skills = {"python", "sql", "ml"}


def team_skills(team):
    return set().union(*(candidates[name] for name in team))


def valid_team(team):
    team = set(team)
    has_required_skills = required_skills <= team_skills(team)
    has_exactly_one_anchor = ("Nia" in team) ^ ("Omar" in team)
    return has_required_skills and has_exactly_one_anchor


def valid_teams(size=4):
    return (
        team
        for team in it.combinations(candidates, size)
        if valid_team(team)
    )

teams = list(valid_teams(4))
len(teams), teams[:10]


(18,
 [('Nia', 'Pia', 'Quin', 'Rae'),
  ('Nia', 'Pia', 'Quin', 'Sol'),
  ('Nia', 'Pia', 'Quin', 'Tao'),
  ('Nia', 'Pia', 'Rae', 'Sol'),
  ('Nia', 'Pia', 'Sol', 'Tao'),
  ('Nia', 'Quin', 'Rae', 'Sol'),
  ('Nia', 'Quin', 'Rae', 'Tao'),
  ('Nia', 'Quin', 'Sol', 'Tao'),
  ('Nia', 'Rae', 'Sol', 'Tao'),
  ('Omar', 'Pia', 'Quin', 'Rae')])

In [27]:
assert len(teams) == 18
assert all(len(team) == 4 for team in teams)
assert all(valid_team(team) for team in teams)
print("Problem 9 tests passed.")


Problem 9 tests passed.


### Extra example: score teams by skill coverage


In [28]:
ranked_teams = sorted(
    ((team, len(team_skills(team)), sorted(team_skills(team))) for team in valid_teams(4)),
    key=lambda row: (-row[1], row[0]),
)

for team, coverage_count, skills in ranked_teams[:8]:
    print(f"{team}: {coverage_count} skills -> {skills}")


('Nia', 'Pia', 'Quin', 'Rae'): 5 skills -> ['ml', 'python', 'sql', 'stats', 'viz']
('Nia', 'Pia', 'Quin', 'Sol'): 5 skills -> ['ml', 'python', 'sql', 'stats', 'viz']
('Nia', 'Pia', 'Quin', 'Tao'): 5 skills -> ['ml', 'python', 'sql', 'stats', 'viz']
('Nia', 'Pia', 'Sol', 'Tao'): 5 skills -> ['ml', 'python', 'sql', 'stats', 'viz']
('Nia', 'Quin', 'Rae', 'Sol'): 5 skills -> ['ml', 'python', 'sql', 'stats', 'viz']
('Nia', 'Quin', 'Rae', 'Tao'): 5 skills -> ['ml', 'python', 'sql', 'stats', 'viz']
('Nia', 'Rae', 'Sol', 'Tao'): 5 skills -> ['ml', 'python', 'sql', 'stats', 'viz']
('Omar', 'Pia', 'Quin', 'Rae'): 5 skills -> ['ml', 'python', 'sql', 'stats', 'viz']


## Problem 10 — Combinations with replacement: recipe scoops

A smoothie shop has four add-ins:

```python
add_ins = ('chia', 'flax', 'protein', 'cacao')
```

A customer chooses exactly 3 scoops. Repetition is allowed and order does not matter.

Tasks:

1. Generate all scoop combinations.
2. Count how many options exist.
3. Filter combinations that contain at least one `'protein'` scoop.


### Solution 10


In [29]:
add_ins = ("chia", "flax", "protein", "cacao")

all_scoop_combos = list(it.combinations_with_replacement(add_ins, 3))
protein_combos = [combo for combo in all_scoop_combos if "protein" in combo]

all_scoop_combos, len(all_scoop_combos), protein_combos[:8], len(protein_combos)


([('chia', 'chia', 'chia'),
  ('chia', 'chia', 'flax'),
  ('chia', 'chia', 'protein'),
  ('chia', 'chia', 'cacao'),
  ('chia', 'flax', 'flax'),
  ('chia', 'flax', 'protein'),
  ('chia', 'flax', 'cacao'),
  ('chia', 'protein', 'protein'),
  ('chia', 'protein', 'cacao'),
  ('chia', 'cacao', 'cacao'),
  ('flax', 'flax', 'flax'),
  ('flax', 'flax', 'protein'),
  ('flax', 'flax', 'cacao'),
  ('flax', 'protein', 'protein'),
  ('flax', 'protein', 'cacao'),
  ('flax', 'cacao', 'cacao'),
  ('protein', 'protein', 'protein'),
  ('protein', 'protein', 'cacao'),
  ('protein', 'cacao', 'cacao'),
  ('cacao', 'cacao', 'cacao')],
 20,
 [('chia', 'chia', 'protein'),
  ('chia', 'flax', 'protein'),
  ('chia', 'protein', 'protein'),
  ('chia', 'protein', 'cacao'),
  ('flax', 'flax', 'protein'),
  ('flax', 'protein', 'protein'),
  ('flax', 'protein', 'cacao'),
  ('protein', 'protein', 'protein')],
 10)

In [30]:
# Number of combinations with replacement: C(n+r-1, r)
assert len(all_scoop_combos) == math.comb(len(add_ins) + 3 - 1, 3) == 20
assert len(protein_combos) == 10
assert all("protein" in combo for combo in protein_combos)
print("Problem 10 tests passed.")


Problem 10 tests passed.


### Extra example: convert scoop tuples to count dictionaries


In [31]:
def combo_to_counts(combo):
    counts = Counter(combo)
    return {name: counts[name] for name in add_ins if counts[name]}

pprint([combo_to_counts(combo) for combo in protein_combos[:10]])


[{'chia': 2, 'protein': 1},
 {'chia': 1, 'flax': 1, 'protein': 1},
 {'chia': 1, 'protein': 2},
 {'cacao': 1, 'chia': 1, 'protein': 1},
 {'flax': 2, 'protein': 1},
 {'flax': 1, 'protein': 2},
 {'cacao': 1, 'flax': 1, 'protein': 1},
 {'protein': 3},
 {'cacao': 1, 'protein': 2},
 {'cacao': 2, 'protein': 1}]


## Problem 11 — Card deck odds using `product` and `combinations`

Create a 52-card deck using `itertools.product`, where each card is a named tuple:

```python
Card(rank, suit)
```

Then compute the exact probability that a 4-card hand consists entirely of aces.

Do it two ways:

1. Brute force with `itertools.combinations`.
2. Direct formula with `math.comb`.


### Solution 11


In [32]:
Card = namedtuple("Card", "rank suit")
SUITS = "SHDC"
RANKS = tuple(map(str, range(2, 11))) + tuple("JQKA")


def make_deck():
    return (Card(rank, suit) for suit, rank in it.product(SUITS, RANKS))


def is_all_aces(hand):
    return all(card.rank == "A" for card in hand)


def brute_force_all_aces_probability():
    total = 0
    acceptable = 0
    for hand in it.combinations(make_deck(), 4):
        total += 1
        if is_all_aces(hand):
            acceptable += 1
    return Fraction(acceptable, total)


def formula_all_aces_probability():
    return Fraction(math.comb(4, 4), math.comb(52, 4))

brute = brute_force_all_aces_probability()
formula = formula_all_aces_probability()
brute, formula, float(formula)


(Fraction(1, 270725), Fraction(1, 270725), 3.6937852063902484e-06)

In [33]:
assert brute == formula == Fraction(1, 270725)
print(f"Probability = {formula} ≈ {float(formula):.10f}")
print("Problem 11 tests passed.")


Probability = 1/270725 ≈ 0.0000036938
Problem 11 tests passed.


### Extra example: preview the generated deck


In [34]:
take(12, make_deck())


[Card(rank='2', suit='S'),
 Card(rank='3', suit='S'),
 Card(rank='4', suit='S'),
 Card(rank='5', suit='S'),
 Card(rank='6', suit='S'),
 Card(rank='7', suit='S'),
 Card(rank='8', suit='S'),
 Card(rank='9', suit='S'),
 Card(rank='10', suit='S'),
 Card(rank='J', suit='S'),
 Card(rank='Q', suit='S'),
 Card(rank='K', suit='S')]

## Problem 12 — Poker hand classification by formulas and a small brute-force check

For a 5-card poker hand from a standard 52-card deck, compute the exact probability of **exactly one pair**.

Use the formula:

1. choose the rank of the pair: `C(13, 1)`
2. choose the 2 suits for that pair: `C(4, 2)`
3. choose 3 other ranks from the remaining 12: `C(12, 3)`
4. choose one suit for each of those 3 ranks: `4^3`
5. divide by all 5-card hands: `C(52, 5)`

Then verify the classification logic on a reduced deck by brute force.


### Solution 12


In [35]:
def exactly_one_pair_count():
    return math.comb(13, 1) * math.comb(4, 2) * math.comb(12, 3) * (4 ** 3)


def total_poker_hands():
    return math.comb(52, 5)


def exactly_one_pair_probability():
    return Fraction(exactly_one_pair_count(), total_poker_hands())

pair_prob = exactly_one_pair_probability()
pair_prob, float(pair_prob)


(Fraction(352, 833), 0.4225690276110444)

In [36]:
def rank_pattern(hand):
    return sorted(Counter(card.rank for card in hand).values(), reverse=True)


def is_exactly_one_pair(hand):
    return rank_pattern(hand) == [2, 1, 1, 1]

# Small brute-force deck: 6 ranks x 2 suits = 12 cards. Choose 5, so only 792 hands.
SMALL_RANKS = tuple("ABCDEF")
SMALL_SUITS = tuple("XY")
small_deck = [Card(rank, suit) for suit, rank in it.product(SMALL_SUITS, SMALL_RANKS)]
small_hands = list(it.combinations(small_deck, 5))
small_pair_count = sum(1 for hand in small_hands if is_exactly_one_pair(hand))

# Formula for small deck with 6 ranks and 2 suits:
# choose pair rank C(6,1), choose both suits C(2,2), choose 3 other ranks C(5,3), choose one suit for each 2^3
small_formula = math.comb(6, 1) * math.comb(2, 2) * math.comb(5, 3) * (2 ** 3)

assert small_pair_count == small_formula == 480
assert pair_prob == Fraction(1098240, 2598960)
print(f"Full-deck exactly-one-pair probability = {pair_prob} ≈ {float(pair_prob):.4f}")
print("Problem 12 tests passed.")


Full-deck exactly-one-pair probability = 352/833 ≈ 0.4226
Problem 12 tests passed.


## Problem 13 — Exhaustive search for valid lock codes

A lock code has length 4. Each position can be one of the digits `0..9`.

A valid code must satisfy:

1. no digit appears more than twice,
2. the code is not strictly increasing,
3. the sum of digits is divisible by 3,
4. the first digit is not `0`.

Use `itertools.product` to count and generate valid codes lazily.


### Solution 13


In [37]:
def valid_code(code):
    digits = tuple(map(int, code))
    return (
        digits[0] != 0
        and max(Counter(digits).values()) <= 2
        and not all(a < b for a, b in zip(digits, digits[1:]))
        and sum(digits) % 3 == 0
    )


def valid_codes():
    return (
        "".join(code)
        for code in it.product("0123456789", repeat=4)
        if valid_code(code)
    )

codes_preview = take(15, valid_codes())
codes_count = consume_count(valid_codes())
codes_preview, codes_count


(['1002',
  '1005',
  '1008',
  '1014',
  '1017',
  '1020',
  '1023',
  '1026',
  '1029',
  '1032',
  '1035',
  '1038',
  '1041',
  '1044',
  '1047'],
 2829)

In [38]:
assert codes_preview[:5] == ['1002', '1005', '1008', '1014', '1017']
assert codes_count == 2829
assert all(valid_code(code) for code in codes_preview)
print("Problem 13 tests passed.")


Problem 13 tests passed.


### Extra example: distribution by first digit


In [39]:
first_digit_counts = Counter(code[0] for code in valid_codes())
for digit, count in sorted(first_digit_counts.items()):
    print(f"first digit {digit}: {count} valid codes")


first digit 1: 303 valid codes
first digit 2: 310 valid codes
first digit 3: 307 valid codes
first digit 4: 318 valid codes
first digit 5: 320 valid codes
first digit 6: 314 valid codes
first digit 7: 321 valid codes
first digit 8: 321 valid codes
first digit 9: 315 valid codes


## Problem 14 — Build a combinations-based feature generator

Suppose a model accepts interaction terms such as:

```python
age × income
age × income × score
```

Write `interaction_terms(features, min_degree, max_degree)` that yields tuples of feature names for all interaction terms from degree `min_degree` to `max_degree`, inclusive.

Use `itertools.combinations`.


### Solution 14


In [40]:
def interaction_terms(features, min_degree=2, max_degree=None):
    features = tuple(features)
    if max_degree is None:
        max_degree = len(features)
    if not (1 <= min_degree <= max_degree <= len(features)):
        raise ValueError("expected 1 <= min_degree <= max_degree <= len(features)")

    for degree in range(min_degree, max_degree + 1):
        yield from it.combinations(features, degree)

features = ("age", "income", "score", "region", "tenure")
terms = list(interaction_terms(features, 2, 3))
terms[:10], len(terms)


([('age', 'income'),
  ('age', 'score'),
  ('age', 'region'),
  ('age', 'tenure'),
  ('income', 'score'),
  ('income', 'region'),
  ('income', 'tenure'),
  ('score', 'region'),
  ('score', 'tenure'),
  ('region', 'tenure')],
 20)

In [41]:
assert len(terms) == math.comb(5, 2) + math.comb(5, 3) == 20
assert terms[0] == ("age", "income")
assert terms[-1] == ("score", "region", "tenure")
print("Problem 14 tests passed.")


Problem 14 tests passed.


### Extra example: format interaction terms for display


In [42]:
formatted_terms = (" × ".join(term) for term in interaction_terms(features, 2, 3))
print("\n".join(take(12, formatted_terms)))


age × income
age × score
age × region
age × tenure
income × score
income × region
income × tenure
score × region
score × tenure
region × tenure
age × income × score
age × income × region


## Problem 15 — Capstone: compare brute-force and formula probabilities

You are rolling five fair six-sided dice. Compute the probability of getting a **full house** pattern: three dice show one value and two dice show another value.

Do it two ways:

1. Brute force with `itertools.product`.
2. Formula.

Formula reasoning:

- choose the value that appears three times: `6`
- choose the value that appears two times: `5`
- choose which 3 of the 5 positions contain the triple value: `C(5, 3)`
- total outcomes: `6^5`


### Solution 15


In [43]:
def is_dice_full_house(outcome):
    return sorted(Counter(outcome).values()) == [2, 3]


def brute_force_dice_full_house_probability():
    total = 0
    acceptable = 0
    for outcome in it.product(range(1, 7), repeat=5):
        total += 1
        if is_dice_full_house(outcome):
            acceptable += 1
    return Fraction(acceptable, total)


def formula_dice_full_house_probability():
    acceptable = 6 * 5 * math.comb(5, 3)
    total = 6 ** 5
    return Fraction(acceptable, total)

full_house_brute = brute_force_dice_full_house_probability()
full_house_formula = formula_dice_full_house_probability()
full_house_brute, full_house_formula, float(full_house_formula)


(Fraction(25, 648), Fraction(25, 648), 0.038580246913580245)

In [44]:
assert full_house_brute == full_house_formula == Fraction(25, 648)
print(f"Probability = {full_house_formula} ≈ {float(full_house_formula):.4f}")
print("Problem 15 tests passed.")


Probability = 25/648 ≈ 0.0386
Problem 15 tests passed.


## Summary of best practices

1. Prefer `itertools.product` over deeply nested loops when you are expressing a Cartesian product.
2. Use `repeat=n` when the same iterable is used multiple times.
3. Remember that `permutations` treats equal values at different positions as distinct elements.
4. Use `combinations` when order does not matter and repetition is not allowed.
5. Use `combinations_with_replacement` when order does not matter and repetition is allowed.
6. Avoid `list(...)` on huge combinatoric iterators; use `islice`, streaming counts, formulas, or filters.
7. Use `Fraction` for exact probabilities.
8. Use formulas such as `math.comb` and `math.factorial` to verify brute-force results.


## Additional drill problems

Try these after studying the solutions above:

1. Generate all 8-character product codes using uppercase letters and digits, but require at least two digits.
2. Count all 7-card hands that contain exactly three aces.
3. Generate all 3D grid points where `x + y + z == 0` using exact `Fraction` coordinates.
4. Count unique permutations of `STATISTICS` without generating them.
5. Build all two-topping and three-topping pizza combinations, allowing repeated toppings only for extra cheese.
6. Compute the probability that four dice contain exactly three unique values.
7. Count circular arrangements of 8 people where two specified people are separated by at least two seats.
8. Create a parameter grid where some parameters are conditional on the selected model type.
